In [3]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from scipy.signal import savgol_filter
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# --- CNN ---
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import TensorDataset, DataLoader
    USE_CNN = True
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"🧠 PyTorch (device: {device})")
except ImportError:
    USE_CNN = False
    print("⚠️ PyTorchなし → CNNスキップ")

# ============================================================
# 1. データ読み込み
# ============================================================
print("📂 データ読み込み中...")
with open('data/train.csv', 'r', encoding='cp932', errors='replace') as f:
    train = pd.read_csv(f)
with open('data/test.csv', 'r', encoding='cp932', errors='replace') as f:
    test = pd.read_csv(f)
submit = pd.read_csv('data/sample_submit.csv', header=None)

train = train[train['樹種'] != 'ベイスギ'].reset_index(drop=True)

spec_cols = [c for c in train.columns
             if c not in ['sample number', 'species number', '樹種', '含水率']]
y_train_log = np.log1p(train['含水率'])
groups = train['species number']

# ============================================================
# 2. 波数・バンドのインデックス定義
# ============================================================
wavenumbers = np.array([float(c) for c in spec_cols])

# --- ピンポイント波数 ---
idx_5150 = np.argmin(np.abs(wavenumbers - 5150))
idx_6900 = np.argmin(np.abs(wavenumbers - 6900))
idx_5900 = np.argmin(np.abs(wavenumbers - 5900))
idx_4300 = np.argmin(np.abs(wavenumbers - 4300))

# --- バンド帯域（面積・平均用）---
BANDS = {
    'water_5150':       (5000, 5300),   # 水 O-H 結合音
    'water_6900':       (6700, 7100),   # 水 O-H 第1倍音
    'cellulose_CH':     (5800, 6000),   # セルロース C-H
    'lignin_arom':      (5900, 6050),   # リグニン 芳香環 C-H
    'lignin_4680':      (4600, 4750),   # リグニン
    'CH_OH_comb':       (4200, 4400),   # C-H + O-H 結合音
}
band_indices = {}
for name, (lo, hi) in BANDS.items():
    band_indices[name] = np.where((wavenumbers >= lo) & (wavenumbers <= hi))[0]

print(f"📏 スペクトル次元数: {len(spec_cols)}")
for name, idx in band_indices.items():
    print(f"  Band [{name}]: {len(idx)} points "
          f"({wavenumbers[idx[0]]:.0f}–{wavenumbers[idx[-1]]:.0f} cm⁻¹)")


# ============================================================
# 3. 前処理関数
# ============================================================

def apply_snv(X):
    m = np.mean(X, axis=1, keepdims=True)
    s = np.std(X, axis=1, keepdims=True) + 1e-8
    return (X - m) / s

def apply_msc(X, ref):
    X_msc = np.zeros_like(X)
    for i in range(X.shape[0]):
        coef = np.polyfit(ref, X[i], 1)
        X_msc[i] = (X[i] - coef[1]) / (coef[0] + 1e-8)
    return X_msc

def apply_derivatives(X):
    d1 = savgol_filter(X, window_length=15, polyorder=2, deriv=1, axis=1)
    d2 = savgol_filter(X, window_length=11, polyorder=2, deriv=2, axis=1)
    return d1, d2


# ============================================================
# 4. 特徴量抽出（考察を全面反映）
# ============================================================

def extract_all_features(X_raw, X_d2, X_d2_raw_only):
    """
    X_raw:        生スペクトル
    X_d2:         SNV → SG二次微分
    X_d2_raw_only: 生 → SG二次微分（SNVなし）— 考察「前処理順序テスト」
    """
    f = {}
    eps = 1e-4
    clip_val = 100.0

    def safe_ratio(num, den):
        den_s = np.where(np.abs(den) < eps, eps, np.abs(den))
        return np.clip(num / den_s, -clip_val, clip_val)

    # ──────────────────────────────────────────
    # 【罠B対策】生スペクトルからの散乱・密度特徴量
    #   SNVで消える情報を事前に保存
    # ──────────────────────────────────────────
    f['raw_mean']      = np.mean(X_raw, axis=1)
    f['raw_std']       = np.std(X_raw, axis=1)
    f['raw_max']       = np.max(X_raw, axis=1)
    f['raw_min']       = np.min(X_raw, axis=1)
    f['raw_range']     = f['raw_max'] - f['raw_min']
    # ベースラインの傾き（散乱の波長依存性 → 粒度/表面情報）
    slopes = np.polyfit(np.arange(X_raw.shape[1]), X_raw.T, 1)[0]
    f['raw_slope']     = slopes

    # ──────────────────────────────────────────
    # 【罠1対策】バンド面積・平均の比率（点→面のロバスト化）
    # ──────────────────────────────────────────
    
    # 各バンドの面積・平均・最大値を計算
    for name, idx in band_indices.items():
        band_d2 = np.abs(X_d2[:, idx])
        f[f'area_{name}']   = np.trapezoid(band_d2, axis=1)
        f[f'mean_{name}']   = np.mean(band_d2, axis=1)
        f[f'max_{name}']    = np.max(band_d2, axis=1)
        
        # SNVなし微分からも
        band_d2r = np.abs(X_d2_raw_only[:, idx])
        f[f'area_raw_{name}']  = np.trapezoid(band_d2r, axis=1)
        f[f'mean_raw_{name}']  = np.mean(band_d2r, axis=1)

    # バンド面積の比率（含水率の直接推定）
    # 水 / セルロース ≈ 含水率
    f['ratio_area_w5150_cell'] = safe_ratio(
        f['area_water_5150'], f['area_cellulose_CH'])
    f['ratio_area_w6900_cell'] = safe_ratio(
        f['area_water_6900'], f['area_cellulose_CH'])
    # 水 / リグニン
    f['ratio_area_w5150_lig']  = safe_ratio(
        f['area_water_5150'], f['area_lignin_arom'])
    # 水 / (セルロース + リグニン) = 水 / 全木材
    f['ratio_area_w5150_wood'] = safe_ratio(
        f['area_water_5150'],
        f['area_cellulose_CH'] + f['area_lignin_arom'] + 1e-8)

    # SNVなし微分でも同様の比率
    f['ratio_area_raw_w5150_cell'] = safe_ratio(
        f['area_raw_water_5150'], f['area_raw_cellulose_CH'])

    # 点ベースの比率も残す（バックアップ）
    f['ratio_d2_5150_5900'] = safe_ratio(X_d2[:, idx_5150], X_d2[:, idx_5900])
    f['ratio_d2_6900_5900'] = safe_ratio(X_d2[:, idx_6900], X_d2[:, idx_5900])
    f['ratio_d2_5150_4300'] = safe_ratio(X_d2[:, idx_5150], X_d2[:, idx_4300])

    # ──────────────────────────────────────────
    # 【罠3対策】ピークシフト検出（Max / Argmax / 波数位置）
    # ──────────────────────────────────────────
    for band_name in ['water_5150', 'water_6900']:
        idx = band_indices[band_name]
        d2_band = np.abs(X_d2[:, idx])
        
        f[f'peak_max_{band_name}']  = np.max(d2_band, axis=1)
        f[f'peak_argmax_{band_name}'] = np.argmax(d2_band, axis=1).astype(float)
        f[f'peak_wn_{band_name}']   = wavenumbers[idx][np.argmax(d2_band, axis=1)]
        
        # ピークの非対称性（左右の面積比）
        argmax_pos = np.argmax(d2_band, axis=1)
        left_area  = np.array([np.sum(d2_band[i, :argmax_pos[i]+1])
                               for i in range(len(d2_band))])
        right_area = np.array([np.sum(d2_band[i, argmax_pos[i]:])
                               for i in range(len(d2_band))])
        f[f'peak_asym_{band_name}'] = safe_ratio(left_area, right_area)

    # 2つの水バンド間のピーク位置差（相対シフト量）
    f['peak_wn_diff_water'] = f['peak_wn_water_5150'] - f['peak_wn_water_6900']

    return pd.DataFrame(f)


# ============================================================
# 5. CNN定義
# ============================================================
if USE_CNN:
    class NIR_CNN(nn.Module):
        def __init__(self, input_dim):
            super().__init__()
            self.conv_block = nn.Sequential(
                nn.Conv1d(1, 32, kernel_size=15, padding=7),
                nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(4), nn.Dropout(0.2),
                nn.Conv1d(32, 64, kernel_size=11, padding=5),
                nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(4), nn.Dropout(0.2),
                nn.Conv1d(64, 128, kernel_size=7, padding=3),
                nn.BatchNorm1d(128), nn.ReLU(), nn.AdaptiveAvgPool1d(8), nn.Dropout(0.3),
            )
            self.fc = nn.Sequential(
                nn.Linear(128 * 8, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, 1)
            )
        def forward(self, x):
            x = x.unsqueeze(1)
            x = self.conv_block(x)
            x = x.view(x.size(0), -1)
            return self.fc(x).squeeze(-1)

    def train_cnn(X_tr, y_tr, X_va, y_va, input_dim, epochs=200, lr=1e-3):
        model = NIR_CNN(input_dim).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        criterion = nn.MSELoss()
        ds = TensorDataset(torch.FloatTensor(X_tr).to(device),
                           torch.FloatTensor(y_tr).to(device))
        loader = DataLoader(ds, batch_size=64, shuffle=True)
        best_loss, best_state, patience = float('inf'), None, 0
        for ep in range(epochs):
            model.train()
            for xb, yb in loader:
                optimizer.zero_grad(); loss = criterion(model(xb), yb)
                loss.backward(); optimizer.step()
            scheduler.step()
            model.eval()
            with torch.no_grad():
                vl = criterion(model(torch.FloatTensor(X_va).to(device)),
                               torch.FloatTensor(y_va).to(device)).item()
            if vl < best_loss:
                best_loss = vl
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1
                if patience >= 20: break
        model.load_state_dict(best_state); model.eval()
        return model


# ============================================================
# 6. 最適ブレンド重み探索
# ============================================================
def find_optimal_weights(pred_list, y_true):
    """OOF予測のリストからRMSE最小のブレンド重みを探索"""
    n = len(pred_list)
    def obj(w):
        blend = sum(w[i] * pred_list[i] for i in range(n))
        return np.sqrt(mean_squared_error(y_true, blend))
    cons = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}
    bounds = [(0, 1)] * n
    res = minimize(obj, [1/n]*n, bounds=bounds, constraints=cons, method='SLSQP')
    return res.x


# ============================================================
# 7. メインCVループ
# ============================================================
print("\n" + "=" * 60)
print("🚀 全考察統合モデル — CVループ開始")
print("=" * 60)

gkf = GroupKFold(n_splits=5)
X_test_raw = test[spec_cols].values

final_lgb = np.zeros(len(test))
final_pls = np.zeros(len(test))
final_pls2 = np.zeros(len(test))  # SNVなし微分PLS
final_rdg = np.zeros(len(test))
final_cnn = np.zeros(len(test)) if USE_CNN else None

# OOF蓄積（最適重み用）
oof_lgb = np.zeros(len(train))
oof_pls = np.zeros(len(train))
oof_pls2 = np.zeros(len(train))
oof_rdg = np.zeros(len(train))
oof_cnn = np.zeros(len(train)) if USE_CNN else None

fold_rmses = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(
    train[spec_cols].values, y_train_log, groups
)):
    print(f"\n{'─'*55}")
    print(f"📁 Fold {fold+1}/5  (train: {len(tr_idx)}, valid: {len(va_idx)})")
    print(f"{'─'*55}")

    X_tr_raw = train[spec_cols].values[tr_idx]
    y_tr = y_train_log.iloc[tr_idx].values
    X_va_raw = train[spec_cols].values[va_idx]
    y_va = y_train_log.iloc[va_idx].values
    X_te_raw = X_test_raw.copy()

    # ── 前処理パス1: SNV → SG微分 ──
    snv_tr = apply_snv(X_tr_raw); snv_va = apply_snv(X_va_raw); snv_te = apply_snv(X_te_raw)
    d1_tr, d2_tr = apply_derivatives(snv_tr)
    d1_va, d2_va = apply_derivatives(snv_va)
    d1_te, d2_te = apply_derivatives(snv_te)

    # ── 前処理パス2: SG微分のみ（SNVなし）── 考察「前処理順序テスト」
    _, d2_raw_tr = apply_derivatives(X_tr_raw)
    _, d2_raw_va = apply_derivatives(X_va_raw)
    _, d2_raw_te = apply_derivatives(X_te_raw)

    # ── MSC ──
    ref = np.mean(X_tr_raw, axis=0)
    msc_tr = apply_msc(X_tr_raw, ref); msc_va = apply_msc(X_va_raw, ref); msc_te = apply_msc(X_te_raw, ref)

    # ── 物理特徴量（全考察統合）──
    phys_tr = extract_all_features(X_tr_raw, d2_tr, d2_raw_tr)
    phys_va = extract_all_features(X_va_raw, d2_va, d2_raw_va)
    phys_te = extract_all_features(X_te_raw, d2_te, d2_raw_te)

    # ── PCA（KNN用は8次元、LGB用は15次元）──
    pca_knn = PCA(n_components=8, random_state=42)
    pca_knn_tr = pca_knn.fit_transform(snv_tr)
    pca_knn_va = pca_knn.transform(snv_va)
    pca_knn_te = pca_knn.transform(snv_te)

    pca_full = PCA(n_components=15, random_state=42)
    pca_full_tr = pca_full.fit_transform(snv_tr)
    pca_full_va = pca_full.transform(snv_va)
    pca_full_te = pca_full.transform(snv_te)

    # ── KNN特徴量（8次元PCA空間）──
    knn = NearestNeighbors(n_neighbors=10, metric='cosine')
    knn.fit(pca_knn_tr)
    dist_tr, ind_tr = knn.kneighbors(pca_knn_tr, n_neighbors=11)
    knn_ym_tr = np.mean(y_tr[ind_tr[:, 1:]], axis=1).reshape(-1, 1)
    knn_ys_tr = np.std(y_tr[ind_tr[:, 1:]], axis=1).reshape(-1, 1)
    knn_dm_tr = np.mean(dist_tr[:, 1:], axis=1).reshape(-1, 1)
    
    dist_va, ind_va = knn.kneighbors(pca_knn_va, n_neighbors=10)
    knn_ym_va = np.mean(y_tr[ind_va], axis=1).reshape(-1, 1)
    knn_ys_va = np.std(y_tr[ind_va], axis=1).reshape(-1, 1)
    knn_dm_va = np.mean(dist_va, axis=1).reshape(-1, 1)
    
    dist_te, ind_te = knn.kneighbors(pca_knn_te, n_neighbors=10)
    knn_ym_te = np.mean(y_tr[ind_te], axis=1).reshape(-1, 1)
    knn_ys_te = np.std(y_tr[ind_te], axis=1).reshape(-1, 1)
    knn_dm_te = np.mean(dist_te, axis=1).reshape(-1, 1)

    # ============ 特徴量数チェック ============
    knn_block_tr = np.hstack([knn_ym_tr, knn_ys_tr, knn_dm_tr])
    knn_block_va = np.hstack([knn_ym_va, knn_ys_va, knn_dm_va])
    knn_block_te = np.hstack([knn_ym_te, knn_ys_te, knn_dm_te])

    if fold == 0:
        print(f"\n  📐 特徴量次元チェック:")
        print(f"     SNV:           {snv_tr.shape[1]}")
        print(f"     d1 (SNV→SG1):  {d1_tr.shape[1]}")
        print(f"     d2 (SNV→SG2):  {d2_tr.shape[1]}")
        print(f"     d2_raw (SG2):  {d2_raw_tr.shape[1]}")
        print(f"     PCA (KNN用):   {pca_knn_tr.shape[1]}")
        print(f"     PCA (LGB用):   {pca_full_tr.shape[1]}")
        print(f"     KNN特徴量:     {knn_block_tr.shape[1]}")
        print(f"     物理特徴量:    {phys_tr.shape[1]}")
        print(f"     物理特徴量カラム: {list(phys_tr.columns)}")

    # ───────────────────────────────────────────
    # モデル1: LightGBM
    # ───────────────────────────────────────────
    feat_tr_lgb = np.hstack([snv_tr, d1_tr, pca_full_tr, knn_block_tr, phys_tr.values])
    feat_va_lgb = np.hstack([snv_va, d1_va, pca_full_va, knn_block_va, phys_va.values])
    feat_te_lgb = np.hstack([snv_te, d1_te, pca_full_te, knn_block_te, phys_te.values])
    
    if fold == 0:
        print(f"     LightGBM入力:  {feat_tr_lgb.shape[1]}")

    lgb_model = lgb.LGBMRegressor(
        n_estimators=2000, learning_rate=0.02, max_depth=5,
        num_leaves=31, subsample=0.7, colsample_bytree=0.2,
        min_child_samples=20, reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, verbosity=-1
    )
    lgb_model.fit(feat_tr_lgb, y_tr,
                  eval_set=[(feat_va_lgb, y_va)],
                  callbacks=[lgb.early_stopping(50, verbose=False)])
    p_va_lgb = np.expm1(lgb_model.predict(feat_va_lgb))
    p_te_lgb = np.expm1(lgb_model.predict(feat_te_lgb))

    # ───────────────────────────────────────────
    # モデル2: PLS（SNV → SG2）
    # ───────────────────────────────────────────
    pls_model = PLSRegression(n_components=8)
    pls_model.fit(d2_tr, y_tr)
    p_va_pls = np.expm1(pls_model.predict(d2_va).flatten())
    p_te_pls = np.expm1(pls_model.predict(d2_te).flatten())

    # ───────────────────────────────────────────
    # モデル2b: PLS（SNVなし → SG2） — 考察「前処理順序テスト」
    # ───────────────────────────────────────────
    pls_model2 = PLSRegression(n_components=8)
    pls_model2.fit(d2_raw_tr, y_tr)
    p_va_pls2 = np.expm1(pls_model2.predict(d2_raw_va).flatten())
    p_te_pls2 = np.expm1(pls_model2.predict(d2_raw_te).flatten())

    # ───────────────────────────────────────────
    # モデル3: Ridge
    # ───────────────────────────────────────────
    feat_tr_rdg = np.hstack([pca_full_tr, knn_block_tr, phys_tr.values])
    feat_va_rdg = np.hstack([pca_full_va, knn_block_va, phys_va.values])
    feat_te_rdg = np.hstack([pca_full_te, knn_block_te, phys_te.values])

    if fold == 0:
        print(f"     Ridge入力:     {feat_tr_rdg.shape[1]}")

    lo = np.percentile(feat_tr_rdg, 1, axis=0)
    hi = np.percentile(feat_tr_rdg, 99, axis=0)
    feat_tr_rdg = np.clip(feat_tr_rdg, lo, hi)
    feat_va_rdg = np.clip(feat_va_rdg, lo, hi)
    feat_te_rdg = np.clip(feat_te_rdg, lo, hi)

    scaler_rdg = StandardScaler()
    feat_tr_rdg_s = scaler_rdg.fit_transform(feat_tr_rdg)
    feat_va_rdg_s = scaler_rdg.transform(feat_va_rdg)
    feat_te_rdg_s = scaler_rdg.transform(feat_te_rdg)

    rdg_model = Ridge(alpha=10.0)
    rdg_model.fit(feat_tr_rdg_s, y_tr)
    p_va_rdg = np.expm1(np.clip(rdg_model.predict(feat_va_rdg_s), 0, 6.5))
    p_te_rdg = np.expm1(np.clip(rdg_model.predict(feat_te_rdg_s), 0, 6.5))

    # ───────────────────────────────────────────
    # モデル4: CNN
    # ───────────────────────────────────────────
    if USE_CNN:
        sc_cnn = StandardScaler()
        cnn_tr = sc_cnn.fit_transform(snv_tr)
        cnn_va = sc_cnn.transform(snv_va)
        cnn_te = sc_cnn.transform(snv_te)
        cnn_model = train_cnn(cnn_tr, y_tr, cnn_va, y_va,
                              input_dim=cnn_tr.shape[1], epochs=200, lr=1e-3)
        with torch.no_grad():
            p_va_cnn = np.expm1(cnn_model(torch.FloatTensor(cnn_va).to(device)).cpu().numpy())
            p_te_cnn = np.expm1(cnn_model(torch.FloatTensor(cnn_te).to(device)).cpu().numpy())

    # ── OOF蓄積 ──
    oof_lgb[va_idx] = p_va_lgb
    oof_pls[va_idx] = p_va_pls
    oof_pls2[va_idx] = p_va_pls2
    oof_rdg[va_idx] = p_va_rdg
    if USE_CNN:
        oof_cnn[va_idx] = p_va_cnn

    final_lgb  += p_te_lgb / 5
    final_pls  += p_te_pls / 5
    final_pls2 += p_te_pls2 / 5
    final_rdg  += p_te_rdg / 5
    if USE_CNN:
        final_cnn += p_te_cnn / 5

    # ── Fold内RMSE表示 ──
    y_va_real = np.expm1(y_va)
    rmse_lgb = np.sqrt(mean_squared_error(y_va_real, p_va_lgb))
    rmse_pls = np.sqrt(mean_squared_error(y_va_real, p_va_pls))
    rmse_pls2 = np.sqrt(mean_squared_error(y_va_real, p_va_pls2))
    rmse_rdg = np.sqrt(mean_squared_error(y_va_real, p_va_rdg))
    print(f"  LGB       RMSE: {rmse_lgb:.4f}")
    print(f"  PLS(SNV)  RMSE: {rmse_pls:.4f}")
    print(f"  PLS(raw)  RMSE: {rmse_pls2:.4f}")
    print(f"  Ridge     RMSE: {rmse_rdg:.4f}")
    if USE_CNN:
        rmse_cnn = np.sqrt(mean_squared_error(y_va_real, p_va_cnn))
        print(f"  CNN       RMSE: {rmse_cnn:.4f}")


# ============================================================
# 8. 最適ブレンド重み探索（OOFベース）
# ============================================================
print(f"\n{'='*60}")
print("🔍 OOFベースの最適ブレンド重み探索")
print(f"{'='*60}")

y_true_real = np.expm1(y_train_log)

if USE_CNN:
    pred_list = [oof_lgb, oof_pls, oof_pls2, oof_rdg, oof_cnn]
    final_list = [final_lgb, final_pls, final_pls2, final_rdg, final_cnn]
    model_names = ['LGB', 'PLS(SNV)', 'PLS(raw)', 'Ridge', 'CNN']
else:
    pred_list = [oof_lgb, oof_pls, oof_pls2, oof_rdg]
    final_list = [final_lgb, final_pls, final_pls2, final_rdg]
    model_names = ['LGB', 'PLS(SNV)', 'PLS(raw)', 'Ridge']

opt_w = find_optimal_weights(pred_list, y_true_real)

print("  最適重み:")
for name, w in zip(model_names, opt_w):
    print(f"    {name:12s}: {w:.4f}")

# 最適重みブレンド
oof_blend = sum(opt_w[i] * pred_list[i] for i in range(len(opt_w)))
oof_rmse = np.sqrt(mean_squared_error(y_true_real, oof_blend))
print(f"\n  📊 最適ブレンド OOF RMSE: {oof_rmse:.4f}")

# 個別モデルのOOF RMSE
for name, oof in zip(model_names, pred_list):
    r = np.sqrt(mean_squared_error(y_true_real, oof))
    print(f"  📊 {name:12s} OOF RMSE: {r:.4f}")


# ============================================================
# 9. 提出ファイル
# ============================================================
final_blend = sum(opt_w[i] * final_list[i] for i in range(len(opt_w)))
final_blend = np.clip(final_blend, 0, None)

submit[1] = final_blend
output_filename = 'submission_full_analysis.csv'
submit.to_csv(output_filename, index=False, header=False)

print(f"\n✅ 提出ファイル: {output_filename}")
print(f"📈 予測統計: min={final_blend.min():.1f}%, "
      f"median={np.median(final_blend):.1f}%, max={final_blend.max():.1f}%")

🧠 PyTorch (device: cuda)
📂 データ読み込み中...
📏 スペクトル次元数: 1555
  Band [water_5150]: 78 points (5300–5003 cm⁻¹)
  Band [water_6900]: 103 points (7097–6704 cm⁻¹)
  Band [cellulose_CH]: 52 points (5998–5801 cm⁻¹)
  Band [lignin_arom]: 39 points (6048–5901 cm⁻¹)
  Band [lignin_4680]: 39 points (4748–4602 cm⁻¹)
  Band [CH_OH_comb]: 52 points (4397–4200 cm⁻¹)

🚀 全考察統合モデル — CVループ開始

───────────────────────────────────────────────────────
📁 Fold 1/5  (train: 940, valid: 270)
───────────────────────────────────────────────────────

  📐 特徴量次元チェック:
     SNV:           1555
     d1 (SNV→SG1):  1555
     d2 (SNV→SG2):  1555
     d2_raw (SG2):  1555
     PCA (KNN用):   8
     PCA (LGB用):   15
     KNN特徴量:     3
     物理特徴量:    53
     物理特徴量カラム: ['raw_mean', 'raw_std', 'raw_max', 'raw_min', 'raw_range', 'raw_slope', 'area_water_5150', 'mean_water_5150', 'max_water_5150', 'area_raw_water_5150', 'mean_raw_water_5150', 'area_water_6900', 'mean_water_6900', 'max_water_6900', 'area_raw_water_6900', 'mean_raw_water

In [4]:
# ============================================================
# 診断コード：原因の切り分け
# ============================================================
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error

# --- 既存のOOF予測があることを前提 ---
# oof_lgb, oof_pls, oof_rdg, oof_cnn, y_true_real が存在する想定

# ──────────────────────────────────────────
# 診断1: 様々な手動重みでOOF RMSEを確認
# ──────────────────────────────────────────
print("=" * 60)
print("診断1: 手動重みの比較")
print("=" * 60)

weight_sets = {
    "元コード風 (LGB重視)":    [0.55, 0.20, 0.10, 0.15],
    "LGBさらに重視":           [0.65, 0.15, 0.05, 0.15],
    "PLS重視":                 [0.25, 0.45, 0.10, 0.20],
    "均等":                    [0.25, 0.25, 0.25, 0.25],
    "LGB + PLS のみ":          [0.50, 0.50, 0.00, 0.00],
    "LGB単独":                 [1.00, 0.00, 0.00, 0.00],
    "PLS単独":                 [0.00, 1.00, 0.00, 0.00],
    "最適化結果":              [0.21, 0.46, 0.09, 0.24],
}

for name, w in weight_sets.items():
    blend = w[0]*oof_lgb + w[1]*oof_pls + w[2]*oof_rdg + w[3]*oof_cnn
    rmse = np.sqrt(mean_squared_error(y_true_real, blend))
    print(f"  {name:25s}  OOF RMSE: {rmse:.4f}")

# ──────────────────────────────────────────
# 診断2: Fold別の樹種確認
# ──────────────────────────────────────────
print("\n" + "=" * 60)
print("診断2: 各Foldの検証樹種")
print("=" * 60)

from sklearn.model_selection import GroupKFold
gkf = GroupKFold(n_splits=5)
for fold, (tr_idx, va_idx) in enumerate(gkf.split(
    train[spec_cols].values, y_train_log, groups
)):
    va_species = train.iloc[va_idx]['樹種'].unique()
    tr_species = train.iloc[tr_idx]['樹種'].unique()
    print(f"\n  Fold {fold+1}:")
    print(f"    検証樹種 ({len(va_species)}): {list(va_species)}")
    print(f"    訓練樹種 ({len(tr_species)}): {list(tr_species)}")

# ──────────────────────────────────────────
# 診断3: PCA累積寄与率
# ──────────────────────────────────────────
print("\n" + "=" * 60)
print("診断3: PCA累積寄与率")
print("=" * 60)

X_all_snv = apply_snv(train[spec_cols].values)
pca_diag = PCA(n_components=20, random_state=42)
pca_diag.fit(X_all_snv)

cum_var = np.cumsum(pca_diag.explained_variance_ratio_)
for i in range(20):
    print(f"  PC{i+1:2d}: 個別 {pca_diag.explained_variance_ratio_[i]*100:.2f}%  "
          f"累積 {cum_var[i]*100:.2f}%")

# ──────────────────────────────────────────
# 診断4: 各PCと含水率の相関
# ──────────────────────────────────────────
print("\n" + "=" * 60)
print("診断4: PC成分と含水率の相関")
print("=" * 60)

pca_scores = pca_diag.transform(X_all_snv)
y_real = train['含水率'].values

for i in range(15):
    corr = np.corrcoef(pca_scores[:, i], y_real)[0, 1]
    print(f"  PC{i+1:2d}: r = {corr:+.4f}  |{'█' * int(abs(corr)*50)}")

# ──────────────────────────────────────────
# 診断5: 特徴量の総次元数
# ──────────────────────────────────────────
print("\n" + "=" * 60)
print("診断5: 特徴量の総次元数")
print("=" * 60)

dims = {
    "SNV":          1555,
    "d1":           1555,
    "PCA_full(15)": 15,
    "KNN(3)":       3,
    "Physics(53)":  53,
}
print(f"  LightGBM入力: {sum(dims.values())} 次元")
print(f"    内訳:")
for k, v in dims.items():
    print(f"      {k:20s}: {v:5d}  ({v/sum(dims.values())*100:.1f}%)")

print(f"\n  Ridge入力:    {15+3+53} 次元")
print(f"  PLS入力:      1555 次元 (d2)")
print(f"  CNN入力:      1555 次元 (SNV scaled)")

診断1: 手動重みの比較
  元コード風 (LGB重視)              OOF RMSE: 13.9310
  LGBさらに重視                   OOF RMSE: 14.5366
  PLS重視                      OOF RMSE: 12.7151
  均等                         OOF RMSE: 13.1462
  LGB + PLS のみ               OOF RMSE: 13.6197
  LGB単独                      OOF RMSE: 17.6185
  PLS単独                      OOF RMSE: 15.5819
  最適化結果                      OOF RMSE: 12.6826

診断2: 各Foldの検証樹種

  Fold 1:
    検証樹種 (2): ['ウエンジ', 'トチ']
    訓練樹種 (10): ['イチョウ', 'ウォールナット', 'クリ', 'スプルース', 'チェリー', 'ナラ', 'ヒノキ', '米ヒバ', 'ベイマツ', 'ホワイトオーク']

  Fold 2:
    検証樹種 (2): ['チェリー', 'ヒノキ']
    訓練樹種 (10): ['イチョウ', 'ウエンジ', 'ウォールナット', 'クリ', 'スプルース', 'トチ', 'ナラ', '米ヒバ', 'ベイマツ', 'ホワイトオーク']

  Fold 3:
    検証樹種 (2): ['ウォールナット', 'クリ']
    訓練樹種 (10): ['イチョウ', 'ウエンジ', 'スプルース', 'チェリー', 'トチ', 'ナラ', 'ヒノキ', '米ヒバ', 'ベイマツ', 'ホワイトオーク']

  Fold 4:
    検証樹種 (3): ['ナラ', 'ベイマツ', 'ホワイトオーク']
    訓練樹種 (9): ['イチョウ', 'ウエンジ', 'ウォールナット', 'クリ', 'スプルース', 'チェリー', 'トチ', 'ヒノキ', '米ヒバ']

  Fold 5:
    検証樹種 (3): ['イチョウ', 'スプルース', '米ヒバ